In [ ]:
import sys
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching")
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching\demo-code\2d")
import torch
torch.set_default_device("cuda")
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import h5py
from collections import defaultdict, Counter
import numpy as np
from rich import print

from flow_model import HCDFlowResMLP, HCDFlow
from metrics import sa,l1, l2, pcc

In [136]:
i_1 = torch.tensor([[1, 2, 3], 
                    [3, 4, 5]], dtype=torch.float64)
i_2 = torch.tensor([[2, 2, 4],
                    [3, 3, 6]], dtype=torch.float64)

In [135]:
i_1.unsqueeze(0).expand(2, -1 , -1)

tensor([[[0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.]]], device='cuda:0', dtype=torch.float64)

In [4]:
(i_2 - i_1).abs().sum(dim=1).max()

tensor(2., device='cuda:0', dtype=torch.float64)

# Load pretrain model

In [7]:
model_path = r"E:\Dai hoc\2526I\dacn\flow-matching\run_real_data\models\model_6e_1024b.pth"
model = HCDFlowResMLP(noise_dim=174, pep_dim=256, time_dim=128, charge_dim=128, num_blocks=12)
model.load_state_dict(torch.load(model_path))

model.eval()

HCDFlowResMLP(
  (cond_embedding): ConditionEmbedding(
    (pep_embedding): Embedding(22, 256, padding_idx=0)
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=1024, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (mlp): ResMLPWithConditioning(
    (blocks): ModuleList(
      (0-11): 12 x ResBlock(
        (norm): RMSNorm((174,), eps=None, elementwise_a

# Test on training data

In [31]:
train_path = r"E:\Dai hoc\2526I\dacn\flow-matching\data\holdout_hcd.hdf5"

In [54]:
with h5py.File(train_path, "r") as f:
    print("Keys:", list(f.keys()))
    seqs = f["sequence_integer"][32:48]
    charges_oh = f["precursor_charge_onehot"][32:48]
    intensities = f["intensities_raw"][32:48]  

Keys:
[
    'collision_energy',
    'collision_energy_aligned',
    'collision_energy_aligned_normed',
    'intensities_raw',
    'masses_pred',
    'masses_raw',
    'method',
    'precursor_charge_onehot',
    'rawfile',
    'reverse',
    'scan_number',
    'score',
    'sequence_integer'
]

In [ ]:
# seqs[0]

array([19, 18, 10,  1, 13,  9, 16, 14,  3, 17,  3, 18, 17, 10,  8, 10, 12,
        9,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

In [ ]:
charges = np.argmax(charges_oh, axis=1) + 1
del charges_oh

In [ ]:
# charges

array([3, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3])

In [ ]:
seq_tensors = torch.tensor(seqs, device="cpu") 
charge_tensors = torch.tensor(charges, device="cpu")
intensity_tensors = torch.tensor(intensities, device="cpu")

dataset = TensorDataset(seq_tensors, charge_tensors, intensity_tensors)


In [ ]:
# intensity_tensors = torch.tensor(intensities, dtype=torch.float32)
# seq_tensors = torch.tensor(seqs, dtype=torch.long)
# charge_tensors = torch.tensor(charges, dtype=torch.float32)

In [59]:
# noise = torch.randn_like(intensity_tensors)


In [ ]:
# with torch.no_grad():
#     res = model.sample(noise, seq_tensors, charge_tensors, step=10)
# # res.requires_grad_(False)

In [62]:
# 10 step on training data

In [ ]:
# sa(res, intensity_tensors)
# (0.39725783467292786, 0.5281283855438232)

(0.42704400420188904, 0.51409512758255)

In [ ]:
# l1(res, intensity_tensors)
# (0.24269579350948334, 51.40113067626953)

(0.23724566400051117, 49.752479553222656)

In [ ]:
# l2(res, intensity_tensors)
# (0.09534946829080582, 23.549968719482422)

(0.09254227578639984, 21.677555084228516)

In [ ]:
# pcc(res, intensity_tensors)
# (0.8293659687042236, 0.8788573145866394)

(0.8404171466827393, 0.8718830347061157)

In [145]:
batch = 16
total_sample = len(seqs)
total_test = 8

loader = DataLoader(
    dataset,
    batch_size=batch,
    shuffle=False,
    pin_memory=True,
    num_workers=4
)

noises = torch.randn(total_test, intensities.shape[1]).unsqueeze(1).expand(total_test, batch, -1)

In [146]:
noises

tensor([[[ 0.5810,  0.1728, -2.9144,  ...,  1.4094,  0.0796,  0.3989],
         [ 0.5810,  0.1728, -2.9144,  ...,  1.4094,  0.0796,  0.3989],
         [ 0.5810,  0.1728, -2.9144,  ...,  1.4094,  0.0796,  0.3989],
         ...,
         [ 0.5810,  0.1728, -2.9144,  ...,  1.4094,  0.0796,  0.3989],
         [ 0.5810,  0.1728, -2.9144,  ...,  1.4094,  0.0796,  0.3989],
         [ 0.5810,  0.1728, -2.9144,  ...,  1.4094,  0.0796,  0.3989]],

        [[ 1.1103,  0.1995,  0.4412,  ..., -0.5901,  1.6657, -0.8698],
         [ 1.1103,  0.1995,  0.4412,  ..., -0.5901,  1.6657, -0.8698],
         [ 1.1103,  0.1995,  0.4412,  ..., -0.5901,  1.6657, -0.8698],
         ...,
         [ 1.1103,  0.1995,  0.4412,  ..., -0.5901,  1.6657, -0.8698],
         [ 1.1103,  0.1995,  0.4412,  ..., -0.5901,  1.6657, -0.8698],
         [ 1.1103,  0.1995,  0.4412,  ..., -0.5901,  1.6657, -0.8698]],

        [[ 0.0616,  0.2238, -1.9138,  ...,  0.9059, -2.3256, -0.1315],
         [ 0.0616,  0.2238, -1.9138,  ...,  0

In [ ]:
l1_losses   = []
l2_losses   = []
sa_losses   = []
pcc_losses  = []
noises = torch.rand(total_test, intensities.shape[1], requires_grad=False)
with torch.no_grad():
    device = torch.get_default_device()
    for seq_b, charge_b, intensity_b in loader:
        seq_b = seq_b.to(device, non_blocking=True)
        charge_b = charge_b.to(device, non_blocking=True)
        intensity_b = intensity_b.to(device, non_blocking=True)
        
        

cuda:0

# Test on test set